<div style="border-left:6px solid #00897B;padding:12px 18px;background:#F5F5F5">
<span style="color:#00897B;font-weight:bold">PAUL MINCHELLA · ONLINE LEARNING · STUDENT TEMPLATE</span>
<h1 style="color:#283593">Learning one day at a time</h1>
<b>FTL, FTRL and OGD on an open real-world demand stream</b><br>
Exercises, implementation contracts and written answers · Suggested duration: 3–4 hours
</div>

This practical accompanies [the course slides](../Cours_Online_Learning.pdf). We forecast daily
Capital Bikeshare rental counts and learn **after**, never before, each outcome is revealed.
The dataset is small enough to inspect every algorithm rather than hide the updates in a library.
OGD is useful here: a low-dimensional, bounded representation with recent-demand information
supports accurate incremental learning. It need not beat every competing method.

### Learning outcomes

By the end, you should be able to:

1. Explore a time-ordered dataset and distinguish known covariates from unavailable information.
2. Implement FTL, full-loss FTRL and projected OGD using one common convex loss and feasible set.
3. Compute the **best fixed hindsight comparator**, including at every completed prefix.
4. Select hyperparameters by **minimum validation average regret**, without selecting on test outcomes.
5. Explain why low average regret does **not** imply small forecast errors, convergence or good tracking.

### Route map

| Part | Content | Suggested time | Course connections (printed slide numbers) |
|:--|:--|--:|:--|
| **0** | Packages, configuration and reproducibility | 10 min | Shared model, loss and optimization |
| **I** | Open data, descriptive statistics, causal features and chronological split | 40 min | Protocol: 5–8, 9, 14 |
| **II** | Geometry, FTL, FTRL, OGD, comparator and validation selection | 90 min | Regret: 26–29; algorithms: 32–49; regression: 53–55 |
| **III** | Final test metrics, diagnostics, guarantees and limits of regret/time | 50 min | Scope: 50–51; ridge: 56–59; inventory: 60–65 |

**Student edition.** Complete the function bodies and write your reasoning in the empty Markdown
answer cells. Function names and signatures are mandatory; their complete docstrings remain in English
in both language editions. A bare `return` is only a placeholder: replace it with the documented output.
Imports, fixed configuration and plotting style are supplied; no learning algorithm or numerical answer is implemented.
Empty Python workspace cells are provided for calls, checks, plots and orchestration. Define your functions first,
then call them in order; the uncompleted notebook intentionally does not run the experiments.
Bind documented shared names in those workspaces (in particular the stream arrays/slices, `FEATURE_SCALE`,
`X_cal`, `y_cal` and `theta_ref`) before dependent calls. Do not change the declared grids or use test outcomes
for selection. You may add cells, but keep the imposed interfaces. Save your own outputs in
`online_learning_bike_sharing_student_results/en`, never in the instructor results directory.
Function docstrings specify required behavior, not a sequence of implementation steps.

The printed slide number is the course footer, not the PDF page number (the PDF contains overlays).

**Important scope.** This is a teaching example chosen for its suitability, not a preregistered
benchmark. The notebook's selection procedure does not access test outcomes. A single historical
stream cannot establish universal superiority or a real-deployment guarantee.


### Student name and date

# Part 0 — Useful packages and reproducibility

## Exercise 0 — Set up a restartable experiment

Import only the packages we need. Use a relative data directory, report versions and choose
one plot style. Why should we restart the kernel and run all cells before sharing the notebook?

The imports, fixed constants and plotting style below are supplied. Complete
`report_environment()` and explain how to check reproducibility. If needed, install dependencies
from the notebook folder with `%pip install -r requirements.txt`,
then restart the kernel. Keep this notebook beside `online_learning_bike_sharing_data` for offline use.


### Your answer — Exercise 0

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import Request, urlopen
from time import perf_counter
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import brentq, minimize
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from IPython.display import display, Markdown

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path.cwd() / "online_learning_bike_sharing_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
ARCHIVE_SHA256 = "b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401"
DAY_SHA256 = "a6bcf826782d3c0fbfdcbeead17cd0884185a0dafe8ff10cd48a874ee7ba18be"

# Declared experiment choices: none is fitted using validation or test outcomes.
TARGET_SCALE = 10_000.0
B = 4.0                      # K = {theta: ||theta||_2 <= B}
N_CAL = 90                   # initial history; unscored
N_VAL = 180                  # scored for hyperparameter selection only
FTRL_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
OGD_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]

# Figure colors; line styles and labels also identify methods.
PALETTE = {
    "teal": "#00897B",
    "skyblue": "#81D4FA",
    "blue": "#42A5F5",
    "darkblue": "#283593",
    "orange": "#FB8C00",
    "purple": "#8E24AA",
}
COLORS = {"FTL": PALETTE["skyblue"], "FTRL": PALETTE["orange"], "OGD": PALETTE["purple"]}
STYLES = {"FTL": "--", "FTRL": "-.", "OGD": "-"}
plt.rcParams.update({
    "figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "axes.titleweight": "bold", "axes.labelcolor": "#333333", "lines.linewidth": 2.0,
    "axes.prop_cycle": plt.cycler(color=[PALETTE["teal"], PALETTE["blue"], PALETTE["orange"]]),
})
pd.set_option("display.max_columns", 18)
pd.set_option("display.width", 150)


def report_environment():
    """
    Report the environment and check the conditions for a clean notebook rerun.

    Returns
    -------
    report : dict
        Keys: "versions" (a pandas Series for Python, NumPy, pandas, Matplotlib
        and SciPy), "data_dir" (Path), "seed" (int) and "figure_style" (dict).
        Display the version table and the resolved data directory as side effects.

    Notes
    -----
    Use the provided imports and fixed configuration. Do not install packages
    inside this function or change experiment choices after observing test results.
    The plotting style and constants above are supplied infrastructure, not an
    exercise solution.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Empirical 95% intervals — scope, dependence and saved arrays

We use a **paired circular moving-block bootstrap**: sample consecutive seven-day
blocks with replacement, wrapping only within the available scored interval.
Each draw resamples the same `(features, outcome, recorded forecasts)` tuples for
every method. This preserves short-range dependence better than an i.i.d. day shuffle.
The 2.5th and 97.5th percentiles of 1,000 resampled statistics form each empirical interval.
Bands have opacity **0.1**.

**What these intervals mean.** They describe sensitivity of the recorded forecast
paths to the empirical resampling scheme. Learners are **not retrained**, selection
is **not repeated**, and the hindsight comparator **is refitted inside every draw**
when computing regret. They are not intervals for the performance of a freshly
retrained adaptive policy. Frequentist coverage would require suitable stochastic
regularity; seasonal drift and this short stream can violate those assumptions.
Treat these as approximate **conditional empirical intervals**, not proven 95% coverage.

**Pointwise, not simultaneous.** Each prefix uses only that prefix, with fresh,
reproducible block draws. We do not use full-period intervals at early dates. Bounds
before 28 observations are `NaN`: four weekly blocks are the minimum declared history.
Looking at many prefixes/candidates does not turn these bands into a confidence
sequence or a multiple-comparison correction. Overlapping intervals are not a test.
Block length 7 is a declared weekly convention, not a data-selected optimum.

Statistical summaries and forecast metrics receive intervals. **Raw observations,
individual recorded predictions/errors, sample counts/ranges, deterministic toy
examples and exact theoretical bounds do not have sampling CIs attached to them.**
Forecast metric intervals must not be mistaken for prediction intervals. The sample
maximum's bootstrap range cannot quantify unseen tail risk. Timing uncertainty is
estimated separately from repeated fresh runs, not resampled calendar days.


In [ ]:
# Reproducible, paired circular moving-block bootstrap.
# These settings describe a sensitivity analysis, not a distribution-free guarantee.
N_BOOT = 1000
BLOCK_LENGTH = 7
MIN_CI_N = 4 * BLOCK_LENGTH
CI_LEVEL = 0.95
CI_ALPHA = 0.10
BOOT_SEED = 202611
N_TIMING_REPEATS = 15
CI_BOUNDS, CI_AXES = {}, {}
MODEL_COLORS = {**COLORS, "Frozen calibration": PALETTE["teal"],
                "Previous day": PALETTE["darkblue"], "Previous week": PALETTE["blue"]}


def block_indices(n, generator, repetitions=N_BOOT, block_length=BLOCK_LENGTH):
    """
    Draw paired circular moving-block bootstrap indices for one scored interval.

    Parameters
    ----------
    n : int
        Positive length of the available interval.
    generator : numpy.random.Generator
        Caller-owned random generator; do not reset it or use global random state.
    repetitions : int, default N_BOOT
        Positive number of bootstrap draws.
    block_length : int, default BLOCK_LENGTH
        Positive consecutive block length; use at most n observations per block.

    Returns
    -------
    indices : ndarray of int, shape (repetitions, n)
        Indices in [0, n), with consecutive order inside each circular block.
        Truncate each concatenated draw to n positions.

    Notes
    -----
    Wrapping must remain inside this interval. Reuse these indices across
    methods to keep comparisons paired; never sample future observations.
    """


    return


def resampling_weights(indices, n):
    """
    Convert bootstrap index rows into observation multiplicities.

    Parameters
    ----------
    indices : ndarray of int, shape (repetitions, sample_length)
        Bootstrap positions taking values in [0, n).
    n : int
        Positive number of original observations.

    Returns
    -------
    weights : ndarray of int, shape (repetitions, n)
        Nonnegative occurrence counts; every row sums to sample_length.
    """


    return


def percentile_ci(samples, axis=0):
    """
    Compute a pointwise empirical percentile interval at CI_LEVEL.

    Parameters
    ----------
    samples : array_like
        Finite bootstrap statistics, possibly with several method/metric axes.
    axis : int, default 0
        Axis indexing independent bootstrap draws.

    Returns
    -------
    bounds : ndarray
        Shape of samples with axis removed, followed by a final dimension of 2.
        The last dimension stores [lower_95, upper_95] in that order.

    Notes
    -----
    These conditional empirical bounds are neither prediction intervals
    nor simultaneous or distribution-free confidence guarantees.
    """


    return


def save_ci(key, bounds, labels=None):
    """
    Validate and register interval bounds together with their axis labels.

    Parameters
    ----------
    key : str
        Unique descriptive registry key.
    bounds : array_like, shape (2,) or (n_points, 2)
        Lower/upper bounds; both endpoints must be NaN together when unavailable.
    labels : iterable or None, default None
        One label per point; use 1-based point indices when omitted.

    Returns
    -------
    registered : ndarray, shape (n_points, 2)
        Two-dimensional float bounds. Save an independent copy in CI_BOUNDS[key]
        and aligned string labels in CI_AXES[key].

    Notes
    -----
    Validate endpoint ordering for finite rows and label count. Do not fill
    early missing intervals with values computed from later observations.
    """


    return


def band(ax, x, bounds, color, label=None):
    """
    Draw a precomputed empirical interval band without changing its bounds.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Destination axes.
    x : array_like, shape (n_points,)
        Numeric or date coordinates aligned with bounds.
    bounds : ndarray, shape (n_points, 2)
        Saved lower/upper endpoints, including any leading NaN rows.
    color : str
        Matplotlib-compatible band color.
    label : str or None, default None
        Optional legend label.

    Returns
    -------
    artist : matplotlib.collections.PolyCollection
        Filled band artist with opacity CI_ALPHA and no fabricated early values.
    """


    return


def rolling_bootstrap(values, window=30, seed=0):
    """
    Compute past-only rolling means and paired block-bootstrap intervals.

    Parameters
    ----------
    values : array_like, shape (n,) or (n, n_series)
        Recorded quantities; square errors before calling if estimating MSE.
    window : int, default 30
        Window size, at least MIN_CI_N; require a full past/current window.
    seed : int, default 0
        Reproducible stream identifier combined with BOOT_SEED and endpoint.

    Returns
    -------
    estimates : ndarray, shape (n, n_series)
        Window means; NaN before the first complete window.
    intervals : ndarray, shape (n, n_series, 2)
        Pointwise paired-bootstrap bounds aligned with estimates.

    Notes
    -----
    Resample independently inside each available trailing window and pair
    columns. This function returns means, not RMSE: the caller applies the
    appropriate transformation to MSE estimates and endpoints.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

# Part I — Descriptive statistics and a causal data stream

## I.1 — Open dataset and provenance

**Dataset:** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset),
DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894), created by Hadi Fanaee-T.
We use **day.csv**: **731 daily observations**, 1 January 2011–31 December 2012,
Capital Bikeshare, Washington, D.C., USA. The hourly file is not used.

The UCI dataset page publishes this dataset under
[Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Credit the dataset and its associated paper:

> Hadi Fanaee-T and João Gama (2013). *Event labeling combining ensemble detectors and
> background knowledge*. Progress in Artificial Intelligence. DOI:
> [10.1007/s13748-013-0040-3](https://doi.org/10.1007/s13748-013-0040-3).

The original `Readme.txt` is bundled unchanged. It predates the current UCI license notice
and requests citation of the paper. The archive and daily CSV are checked against fixed SHA-256
hashes, so a changed download is not silently accepted. We inspect archive members individually;
we do not extract arbitrary archive paths.

## Exercise 1 — Inspect the raw observations

Load the daily data. Check dates, duplicates, missing values, units, and whether
`cnt = casual + registered`. Which columns would leak the answer into a forecast?

Complete the documented loader and integrity audit. Build your own availability table for calendar
information, past counts, same-day weather, past weather, `casual`, `registered`, and `instant`.
Explain what rental counts do and do not measure. Use verified local files when available.


### Your answer — Exercise 1

In [ ]:
def sha256(data):
    """
    Return the SHA-256 hexadecimal digest of a byte payload.

    Parameters
    ----------
    data : bytes
        Exact file or archive bytes to verify.

    Returns
    -------
    digest : str
        Lowercase 64-character hexadecimal SHA-256 digest.
    """


    return


def load_daily_data(data_dir=DATA_DIR):
    """
    Load and validate the original UCI daily Bike Sharing data.

    Parameters
    ----------
    data_dir : pathlib.Path, default DATA_DIR
        Local cache containing day.csv and, optionally, bike_sharing.zip.
        Prefer the local verified CSV; download from DATA_URL only if needed.

    Returns
    -------
    frame : pandas.DataFrame, shape (731, 15)
        Original data indexed by parsed dteday, sorted chronologically. Preserve
        original column names and recorded weather units; dteday is not a column.

    Raises
    ------
    RuntimeError
        A needed download fails; explain how to provide the verified local CSV.
    ValueError
        A checksum differs from its declared value or a data integrity check fails.

    Notes
    -----
    Verify CSV bytes against DAY_SHA256 before parsing. If downloading or
    using a cached archive, verify ARCHIVE_SHA256 and read only the known members
    "day.csv" and "Readme.txt" without arbitrary path extraction. Use a finite
    60-second timeout for remote reads. Cache verified bytes and preserve the
    original README. Check row count, unique consecutive dates, missingness,
    and the count decomposition required by Exercise 1.
    """


    return


def audit_daily_data(daily):
    """
    Summarize data integrity, date coverage and observed units.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw daily data returned by load_daily_data, with a DatetimeIndex.

    Returns
    -------
    audit : pandas.Series
        Row/column counts, first/last dates, duplicate and missing counts,
        minimum/maximum observed rentals, and count-decomposition status.
        Display the audit and the first few rows without fitting a predictor.

    Notes
    -----
    This structural audit is not test-set model selection. Reserve outcome
    relationships and descriptive modelling for the calibration period.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## I.2 — Define the chronology before exploring relationships

The first seven raw days supply lagged history. Of the **724 usable days**, use:

- **90 calibration days:** unscored initialization/history; retained in FTL/FTRL cumulative
  objectives, used through the initial coefficients for OGD;
- **180 validation days:** choose hyperparameters using average regret;
- **454 test days:** final prequential evaluation, with choices frozen.

Only dates define this split. No random train/test shuffle is appropriate for the intended
future-prediction question. Descriptive analysis of outcomes is restricted to calibration data
before selection. The all-period plot below is reserved for **Part III**.

## Exercise 2 — Construct strictly past-based features

Build demand lags at one and seven days and a seven-day trailing mean that excludes today.
Add lagged weather and calendar cycles. Avoid a fitted full-data scaler. How can we make
$\|x_t\|_2\le1$ without looking into the future?

Keep the declared demand scale `TARGET_SCALE = 10_000.0`; it is not an estimated maximum or a capacity.
Use recorded weather units and a fixed 365.25-day annual cycle, retaining leap day.
Justify the common feature divisor and the intercept's units. Explain how a future outcome above
the declared bound would affect the assumptions, without silently clipping it.
In the execution cell, bind the documented `prepare_stream` outputs to the names used later.
Implement and run an independent feature-causality check.


### Your answer — Exercise 2

In [ ]:
def make_features(frame):
    """
    Construct chronological predictors available before each outcome.

    Parameters
    ----------
    frame : pandas.DataFrame
        Raw UCI daily observations indexed by date, including the past history
        needed for lags. Keep the original normalized weather units.

    Returns
    -------
    features : pandas.DataFrame, shape (724, 12) for the full supplied data
        Date-indexed predictors, in this order: count_lag_1, count_lag_7,
        count_mean_past_7, temp_lag_1, hum_lag_1, windspeed_lag_1, annual_sin,
        annual_cos, weekly_sin, weekly_cos, workingday, holiday.
        Drop only rows without the required initial history. Do not add the
        intercept or apply the common vector divisor inside this function.

    Notes
    -----
    Scale demand-derived columns by TARGET_SCALE. Use a fixed 365.25-day
    annual cycle with day_of_year - 1 and a seven-day weekly cycle with Monday=0.
    All predictors must be reconstructible before that day's count is revealed.
    Do not use fitted full-data normalization or realized same-day weather.
    """


    return


def prepare_stream(daily, features):
    """
    Assemble the bounded linear representation and chronological split.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw date-indexed observations.
    features : pandas.DataFrame
        The unscaled predictor table returned by make_features.

    Returns
    -------
    stream : dict
        Required keys: dates (DatetimeIndex), feature_names (list beginning with
        "intercept"), d (int), FEATURE_SCALE (fixed scalar), X (n, d), y (n,),
        cal_slice, val_slice, test_slice (slice objects), N_TEST (int), and
        split_table (DataFrame of dates and sizes). For this dataset n=724.
        X includes an intercept and uses the common fixed divisor justified in
        your written answer; y is the count divided by TARGET_SCALE.

    Notes
    -----
    Use N_CAL and N_VAL, then reserve the remaining observations for test.
    Verify finite values and the stated norm/outcome bounds without clipping
    inconvenient observations. Display the chronological split table. In the
    execution workspace, bind the documented names for later exercises; no
    fitted transformation may use validation or test outcomes.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

In [ ]:
def check_feature_causality(daily, features, probe_date):
    """
    Test that a feature row uses only information available before its day.

    Parameters
    ----------
    daily : pandas.DataFrame
        Original raw observations.
    features : pandas.DataFrame
        Predictor table produced by make_features.
    probe_date : pandas.Timestamp
        A usable date with at least seven preceding daily observations.

    Returns
    -------
    None
        Raise AssertionError if a causality or trailing-window check fails.

    Notes
    -----
    Perturb today's and future counts in a copy, rebuild the predictors and
    compare the probe-date row. Independently verify its trailing mean from
    raw history. Do not mutate daily or features.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## I.3 — Descriptive statistics, using initial history only

## Exercise 3 — Describe before modelling

On the calibration period, report central tendency, spread, missingness and selected correlations.
Plot the demand trajectory, its distribution, weekday differences and its relationship with
previous-day demand. What do these plots suggest, and what do they **not** establish?

Use only calibration outcomes. Attach the declared empirical intervals to inferential summaries,
not to exact sample facts. Discuss initial-season representativeness, confounding, and the limits
of interpreting a lag association. Do not tune features after viewing test performance.


### Your answer — Exercise 3

In [ ]:
def describe_calibration(daily, features, y, dates, cal_slice):
    """
    Produce calibration-only summaries, associations and descriptive plots.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw date-indexed observations.
    features : pandas.DataFrame
        Chronological predictor table.
    y : ndarray, shape (n,)
        Normalized outcomes aligned with features.
    dates : pandas.DatetimeIndex
        Dates aligned with y.
    cal_slice : slice
        Initial unscored period; use no later outcomes in this analysis.

    Returns
    -------
    analysis : dict
        Include calibration (DataFrame), summary (DataFrame), summary_ci
        (DataFrame), weekday_stats (DataFrame), correlation_table (DataFrame),
        and figures (list of Matplotlib figures). Display the requested tables
        and the trajectory/distribution/weekday/previous-demand plots.

    Notes
    -----
    Report central tendency, spread and missingness for cnt, casual,
    registered, temp, hum and windspeed. Add paired block-bootstrap intervals
    for inferential summaries and save all plotted interval arrays before
    plotting. Raw observations and exact sample facts do not need intervals.
    Do not infer causality or future stationarity from descriptive association.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

# Part II — Online algorithms and minimum validation average regret

## II.1 — One common loss, geometry and protocol

For every model, predict normalized rentals with
$$
\widehat y_t=x_t^\top\theta_t,\qquad
\ell_t(\theta)=\tfrac12(x_t^\top\theta-y_t)^2,\qquad
K=\{\theta\in\mathbb R^d:\|\theta\|_2\le B\},\quad B=4.
$$
The same unregularized loss and feasible set define **all** reported regrets.
The regularizer in FTRL chooses decisions; it is **not** added to evaluated loss.

```text
At the start of a scored day:
    construct x_today using the calendar and already observed days
    predict with theta_today; store that parameter and prediction
    reveal y_today and score 0.5 * (prediction - y_today)^2
    update coefficients using that newly revealed outcome
    move to the next day
```

## Exercise 4 — Derive the gradient, curvature and projection

Compute $\nabla\ell_t$, $\nabla^2\ell_t$ and $\Pi_K(z)$.
Is the loss strongly convex in all $d$ parameter directions?

Include zero-vector and feasibility cases in your projection checks, and verify the gradient by finite differences.
Explain whether cumulative full-rank data imply per-observation strong convexity.
As a hand calculation, take $K=[0,1]$, $\theta_1=0.5$, $\eta=1/4$ and outcomes 0.8 then 0.2.
Tabulate both predict–reveal–update rounds without reading the next outcome early.


### Your answer — Exercise 4

In [ ]:
def squared_loss(theta, x, target):
    """
    Evaluate the common unregularized prediction loss for one observation.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficient vector.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.

    Returns
    -------
    loss : float
        Half squared normalized prediction error, with no FTRL penalty.
    """


    return


def squared_gradient(theta, x, target):
    """
    Evaluate the parameter gradient of squared_loss.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Current coefficient vector.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.

    Returns
    -------
    gradient : ndarray, shape (d,)
        Gradient in the same coordinate system as theta; do not add a penalty.
    """


    return


def project_ball(vector, radius=B):
    """
    Project a vector onto a closed Euclidean ball centered at zero.

    Parameters
    ----------
    vector : array_like, shape (d,)
        Candidate coefficient vector.
    radius : float, default B
        Positive radius of the feasible ball.

    Returns
    -------
    projected : ndarray, shape (d,)
        Closest feasible vector in Euclidean distance. Handle the zero vector
        and already feasible inputs without changing the caller's array.
    """


    return


def check_geometry(X, seed=SEED):
    """
    Validate your loss gradient and projection independently.

    Parameters
    ----------
    X : ndarray, shape (n, d)
        Feature matrix containing at least one nonzero row.
    seed : int, default SEED
        Seed for reproducible numerical examples.

    Returns
    -------
    None
        Raise AssertionError if a finite-difference gradient check, Hessian-rank
        check, zero-vector case or projected-norm check fails.

    Notes
    -----
    Design the numerical checks yourself; do not hard-code computed answers.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## II.2 — Solve constrained least squares correctly

## Exercise 5 — Find a leader on the Euclidean ball

Let $A=\sum x_sx_s^\top$ and $b=\sum x_sy_s$. The loss-dependent part of the objective is
$\tfrac12\theta^\top A\theta-b^\top\theta$.
How do we minimize it over $K$, including when the design is rank deficient?

State the KKT system, your tie-breaking convention and numerical tolerances. Handle both active and inactive
constraints, singular designs and zero design. Verify your solver against an independent constrained optimizer.
Construct an example showing whether projection of the unconstrained optimum is sufficient.


### Your answer — Exercise 5

In [ ]:
def solve_ball_quadratic(A, b, radius=B):
    """
    Minimize a convex quadratic over the Euclidean coefficient ball.

    Parameters
    ----------
    A : ndarray, shape (d, d)
        Symmetric positive-semidefinite quadratic matrix, possibly singular.
    b : ndarray, shape (d,)
        Linear coefficient; the objective is 0.5 * theta.T @ A @ theta - b.T @ theta.
    radius : float, default B
        Positive Euclidean radius.

    Returns
    -------
    theta : ndarray, shape (d,)
        Global constrained minimizer. Select the minimum-norm minimizer when
        the unconstrained optimum is nonunique and feasible.

    Raises
    ------
    ValueError
        Matrix dimensions, radius or positive-semidefiniteness are invalid.

    Notes
    -----
    Implement the KKT-based solution you derive in Markdown. Handle
    inactive/active constraints, singular designs, zero design and nonzero
    linear components in numerical null directions. Declare numerical
    tolerances. Do not substitute projection of the unconstrained optimum.
    """


    return


def fit_constrained(X_fit, y_fit, lam=0.0, reference=None):
    """
    Fit constrained least squares with one optional anchored FTRL penalty.

    Parameters
    ----------
    X_fit : ndarray, shape (n, d)
        Features on exactly the intended fitting interval.
    y_fit : ndarray, shape (n,)
        Aligned normalized outcomes.
    lam : float, default 0.0
        Nonnegative weight of one quadratic penalty on the cumulative loss.
    reference : ndarray, shape (d,), or None, default None
        Penalty center; use the zero vector when omitted.

    Returns
    -------
    theta : ndarray, shape (d,)
        Minimizer on the radius-B ball of cumulative half squared losses plus
        lam / 2 times squared distance to reference.

    Notes
    -----
    Use solve_ball_quadratic; no penalty is included when lam=0.
    The intercept is penalized when lam>0, just like all other coordinates.
    """


    return


def check_quadratic_solver(d, seed=SEED):
    """
    Independently test constrained quadratic optimality and edge cases.

    Parameters
    ----------
    d : int
        Positive feature dimension.
    seed : int, default SEED
        Seed for reproducible random numerical checks.

    Returns
    -------
    None
        Raise AssertionError if feasibility, KKT residuals or comparison with
        an independent SciPy constrained optimizer fails.

    Notes
    -----
    Include full-rank, rank-deficient, zero-design, active/inactive and
    anchored-penalty cases. Construct a counterexample to naive projection
    rather than copying an answer from a completed notebook.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## II.3 — FTL, FTRL and OGD

## Exercise 6 — Translate the three definitions into state updates

Use all revealed history for the leaders. Let $\theta_{\rm ref}$ be the constrained
least-squares fit on the calibration period. Start every primary method there.
Write the three post-observation updates and specify the OGD clock.

For OGD use $\eta_k=\eta_0/\sqrt{k}$, with $k=1$ for the first validation update and no reset at test.
All methods start from the calibration-only constrained reference. Preserve the class and method names.
Bind `X_cal`, `y_cal`, `theta_ref`, and `FEATURE_SCALE` in your execution workspace before dependent calls.
Test independent mutable states and equality of the initial predictions.


### Your answer — Exercise 6

In [ ]:
def initialize_calibration(X, y, cal_slice):
    """
    Fit the common feasible calibration reference for all primary methods.

    Parameters
    ----------
    X : ndarray, shape (n, d)
        Chronological bounded features.
    y : ndarray, shape (n,)
        Aligned normalized outcomes.
    cal_slice : slice
        Unscored initial history only.

    Returns
    -------
    X_cal : ndarray, shape (N_CAL, d)
        Calibration features.
    y_cal : ndarray, shape (N_CAL,)
        Calibration outcomes.
    theta_ref : ndarray, shape (d,)
        Unregularized constrained calibration fit.

    Notes
    -----
    Bind X_cal, y_cal and theta_ref in the execution workspace before
    calling new_model. No validation or test observation may enter this fit.
    """


    return


class Leader:
    """Full-loss FTL/FTRL state with A, b, reference, lam and theta attributes."""

    def __init__(self, X_history, y_history, reference, lam=0.0):
        """
        Initialize a leader from calibration history and a fixed penalty center.

        Parameters
        ----------
        X_history : ndarray, shape (n_history, d)
            Already observed initial feature rows.
        y_history : ndarray, shape (n_history,)
            Already observed normalized outcomes.
        reference : ndarray, shape (d,)
            Calibration-only feasible reference; store an independent copy.
        lam : float, default 0.0
            Nonnegative fixed cumulative penalty; zero selects FTL.

        Returns
        -------
        None
            Create A (d, d), b (d,), reference (d,), lam (float) and theta (d,).
            theta must be the current constrained leader, not a future-data fit.

        Notes
        -----
        Store sufficient statistics rather than all historical observations.
        """


        return

    def _fit(self):
        """
        Compute the current full-loss constrained leader without adding data.

        Returns
        -------
        theta : ndarray, shape (d,)
            Minimizer using current A, b, reference and lam on the radius-B ball.

        Notes
        -----
        Do not mutate A or b and do not scale the single penalty by history length.
        """


        return

    def update(self, x, target):
        """
        Incorporate one newly revealed observation and refresh the leader.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Features for the observation whose prediction has already been recorded.
        target : float
            Newly revealed normalized outcome.

        Returns
        -------
        None
            Mutate A, b and theta consistently with the full revealed history.
            Preserve the fixed reference and penalty weight.
        """


        return


class OGD:
    """Projected online-gradient state with theta, eta0 and persistent k attributes."""

    def __init__(self, initial_theta, eta0):
        """
        Initialize projected OGD without replaying calibration as scored updates.

        Parameters
        ----------
        initial_theta : ndarray, shape (d,)
            Feasible calibration coefficient vector; store an independent copy.
        eta0 : float
            Positive scale of the decreasing learning-rate schedule.

        Returns
        -------
        None
            Create theta (d,), eta0 (float), and k (int), initially zero.
        """


        return

    def update(self, x, target):
        """
        Apply one projected OGD update after the current outcome is revealed.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Features for the already-recorded prediction.
        target : float
            Newly revealed normalized outcome.

        Returns
        -------
        None
            Advance k once and replace theta by the next feasible coefficient vector.

        Notes
        -----
        Use the decreasing schedule specified in Exercise 6. The first
        validation update has index 1. Keep the same clock across validation and
        test; never reset it at a plot or scoring boundary.
        """


        return


class FixedPredictor:
    """Fixed-vector baseline exposing the same theta/update protocol."""

    def __init__(self, theta):
        """
        Create a feasible fixed linear predictor.

        Parameters
        ----------
        theta : ndarray, shape (d,)
            Fixed coefficient vector; keep a private copy.

        Returns
        -------
        None
            Create theta without sharing mutable state with the caller.
        """


        return

    def update(self, x, target):
        """
        Accept feedback without changing the fixed predictor.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Observed features, accepted for protocol compatibility only.
        target : float
            Observed outcome, accepted for protocol compatibility only.

        Returns
        -------
        None
            Leave theta unchanged; this method intentionally performs no learning.
        """


        return


def new_model(method, value=0.0):
    """
    Create an independent primary learner at the shared calibration state.

    Parameters
    ----------
    method : {"FTL", "FTRL", "OGD"}
        Requested primary algorithm.
    value : float, default 0.0
        FTRL penalty weight or positive OGD step scale; unused for FTL.

    Returns
    -------
    model : Leader or OGD
        Fresh object exposing theta and update(x, target), initialized using
        only the workspace globals X_cal, y_cal and theta_ref.

    Raises
    ------
    ValueError
        method is unknown or its requested hyperparameter is invalid.

    Notes
    -----
    Each call must own independent mutable state. All primary models
    must have the same first validation prediction. Do not reuse a previously
    validated or tested object when creating a new candidate.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

### Course bridge — Distinguish three quadratic penalties

Build a comparison table for full-loss FTRL, linearized FTRL (slides 40–43), and a per-round ridge loss
(slides 56–59). State the next-decision objective, the evaluated loss and the relevant assumptions.
Explain when linearized FTRL and OGD coincide, and whether projection preserves that equivalence.
Use `compare_projected_updates` on the declared two-gradient example and discuss the result in your Exercise 6 answer.


In [ ]:
def compare_projected_updates(gradients, eta=1.0, initial_theta=0.0):
    """
    Compare projected OGD and quadratic linearized FTRL on a scalar stream.

    Parameters
    ----------
    gradients : array_like, shape (n,)
        Gradients of scalar linear convex losses; use [-2.0, 1.0] for the bridge.
    eta : float, default 1.0
        Positive constant step size.
    initial_theta : float, default 0.0
        Shared initial point in [0, 1] and quadratic penalty center.

    Returns
    -------
    comparison : pandas.DataFrame
        Columns: "After step", "Gradient", "Projected OGD", "Linearized FTRL".
        Include the post-update scalar decision for each step and method.

    Notes
    -----
    Use the feasible interval [0, 1]. This is a separate illustration, not
    the full-loss FTRL implementation used for bike demand.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## II.4 — A test-then-train evaluation engine

## Exercise 7 — Record the parameter that actually made the prediction

Implement a reusable loop. Store predictions, unregularized losses, coefficient vectors and
update magnitudes. Verify that replacing an outcome cannot change that same day's prediction.
Why would scoring *after* fitting on today's outcome be invalid?

Test prefix causality and the identity between FTL and zero-penalty FTRL.
Explain why coefficient snapshots must be copied rather than stored as aliases.


### Your answer — Exercise 7

In [ ]:
def prequential_run(model, X_stream, y_stream):
    """
    Record causal forecasts, then update the same learner along a stream.

    Parameters
    ----------
    model : object
        Mutable learner with theta (d,) and update(x, target).
    X_stream : ndarray, shape (n, d)
        Chronological feature rows.
    y_stream : ndarray, shape (n,)
        Aligned outcomes revealed only after each corresponding prediction.

    Returns
    -------
    result : dict
        Keys: prediction (n,), loss (n,), theta (n, d), movement (n,), seconds
        (float), and model (the same learner, in its final updated state).
        theta records independent copies of pre-update coefficients; movement
        records the norm of each post-minus-pre update. loss is unregularized.

    Notes
    -----
    Predict before reading the current target. Never reset an existing
    learner clock. Validate prediction/recorded-parameter consistency and
    feasibility. Include loop/logging overhead in seconds, but no hindsight
    comparison. Calling this function mutates the passed learner.
    """


    return


def check_prequential_protocol(X_stream, y_stream):
    """
    Test prefix causality, independent states and the zero-penalty identity.

    Parameters
    ----------
    X_stream : ndarray, shape (n, d)
        Short chronological feature stream with n>=4.
    y_stream : ndarray, shape (n,)
        Aligned outcomes for causality tests.

    Returns
    -------
    None
        Raise AssertionError if current/future target perturbations affect
        already-fixed predictions, recorded coefficients are overwritten,
        or FTRL with zero penalty differs from FTL.

    Notes
    -----
    Create fresh models for each test; initialize calibration globals first.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## II.5 — Static regret, with a genuine hindsight optimum

## Exercise 8 — Compute $R_n/n$ on a scored interval

For a scored interval $\mathcal I=(t_0+1,\ldots,t_0+n)$, define
$$
L_n^{\rm alg}=\sum_{j=1}^n\ell_{t_0+j}(\theta_{t_0+j}),\quad
L_n^\star=\min_{u\in K}\sum_{j=1}^n\ell_{t_0+j}(u),\quad
R_n=L_n^{\rm alg}-L_n^\star.
$$
Compute this optimum for every completed prefix. Should calibration observations be included
in the scored comparator? Should the FTRL penalty be included?

State which interval and loss define each comparator. Distinguish one fixed optimum per completed prefix
from an oracle that changes every day. Preserve signed regret and use the realized observed-lag feature rows.


### Your answer — Exercise 8

In [ ]:
def prefix_oracle_losses(X_score, y_score):
    """
    Compute the best feasible fixed unregularized loss on every scored prefix.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features from the scored interval only, not its initialization history.
    y_score : ndarray, shape (n,)
        Aligned scored outcomes.

    Returns
    -------
    optimal_losses : ndarray, shape (n,)
        Entry j is the minimum cumulative loss of one fixed feasible vector
        evaluated on all j+1 observations of that completed prefix.

    Notes
    -----
    Refit the comparator separately at each horizon. Do not sum the
    last-day losses of changing oracle vectors. Use the same feasible ball,
    realized feature rows and unregularized loss as the online learners.
    Only tiny negative loss values from numerical cancellation may be removed;
    negative regret itself must never be clipped.
    """


    return


def regret_curves(result, oracle_losses):
    """
    Compare recorded cumulative learner losses with matching prefix optima.

    Parameters
    ----------
    result : dict
        prequential_run result containing a loss vector of length n.
    oracle_losses : ndarray, shape (n,)
        Exact cumulative fixed-comparator optimum for each same scored prefix.

    Returns
    -------
    regret : ndarray, shape (n,)
        Signed cumulative static regret.
    average_regret : ndarray, shape (n,)
        Signed regret divided by the corresponding positive prefix length.

    Notes
    -----
    Do not take absolute values, add algorithmic penalties or include
    calibration observations in a validation/test comparator.
    """


    return


def check_prefix_oracles(X_score, y_score, horizons=(1, 7, 50, 180)):
    """
    Validate prefix oracle accounting against direct constrained fits.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Scored features.
    y_score : ndarray, shape (n,)
        Scored outcomes.
    horizons : iterable of int, default (1, 7, 50, 180)
        Positive prefix lengths; test only values no greater than n.

    Returns
    -------
    None
        Raise AssertionError if cumulative optimum losses disagree with direct
        whole-prefix evaluations or decrease beyond numerical tolerance.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

### Implementation task — Paired score bootstrap with a refitted oracle

Implement the documented batched and weighted comparator helpers. Each bootstrap draw must have one
fixed constrained, unregularized comparator evaluated over the entire resampled prefix. Check scalar/batched
agreement, including singular and active-boundary cases. Save prefix-only interval arrays before plotting.
Use the scope and minimum-history conventions specified at the beginning of the notebook.


In [ ]:
METRIC_KEYS = {
    "Total loss": "total_loss", "Mean loss": "mean_loss", "Static regret": "regret",
    "Regret / day": "average_regret", "RMSE": "rmse", "MAE": "mae", "Bias": "bias",
    "P95 abs. error": "p95_absolute_error", "Max abs. error": "max_absolute_error",
}


def batched_oracle_losses(A, b, yy, radius=B):
    """
    Solve independent unregularized constrained least-squares oracle problems.

    Parameters
    ----------
    A : ndarray, shape (n_draws, d, d)
        Positive-semidefinite weighted Gram matrices, possibly singular.
    b : ndarray, shape (n_draws, d)
        Matching weighted feature-target cross-products.
    yy : ndarray, shape (n_draws,)
        Matching weighted sums of squared targets.
    radius : float, default B
        Positive shared feasible coefficient radius.

    Returns
    -------
    losses : ndarray, shape (n_draws,)
        Minimum whole-draw half squared loss for one fixed vector per draw.
        Include yy when recovering objective values; no regularization.

    Notes
    -----
    Handle inactive and active constraints, rank deficiency and numerical
    cancellation. The scalar solve_ball_quadratic may serve as a correctness
    reference, but implement a practical batched solver for prefix intervals.
    Never approximate this oracle by clipping an unconstrained solution.
    """


    return


def weighted_oracle_losses(X_score, y_score, weights):
    """
    Fit a separate fixed comparator for each paired bootstrap weight row.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Recorded features for the available scored interval.
    y_score : ndarray, shape (n,)
        Aligned outcomes.
    weights : ndarray, shape (n_draws, n)
        Nonnegative observation multiplicities for each resampled interval.

    Returns
    -------
    losses : ndarray, shape (n_draws,)
        Minimum weighted unregularized cumulative loss on the radius-B ball.

    Notes
    -----
    Keep features and outcomes paired, including observed target lags.
    Do not recursively replace lag columns with newly generated predictions.
    """


    return


def bootstrap_prefix_metrics(X_score, y_score, predictions, phase, seed, axis_labels=None):
    """
    Estimate paired prefix-only metric intervals with refitted fixed comparators.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features for one scored interval; require n>=MIN_CI_N.
    y_score : ndarray, shape (n,)
        Matching normalized targets.
    predictions : dict[str, ndarray]
        Named recorded forecast paths, each of shape (n,).
    phase : str
        Registry prefix, such as "validation" or "test".
    seed : int
        Phase-specific reproducible seed component combined with BOOT_SEED.
    axis_labels : iterable or None, default None
        Optional n labels aligned with scored days.

    Returns
    -------
    bootstrap : dict
        Keys: names (ordered method labels), bounds (method -> metric -> (n, 2)),
        oracle_bounds (metric -> (n, 2)), final_draws (metric -> (N_BOOT, m)),
        final_oracle_draws (metric -> (N_BOOT,)), final_indices (N_BOOT, n).
        Prefix metrics: total_loss, mean_loss, regret, average_regret, rmse.
        Oracle metrics: total_loss, mean_loss, rmse. Final learner draws also
        contain mae, bias, p95_absolute_error and max_absolute_error.
        Error metrics use original rental units; losses use normalized units.

    Raises
    ------
    ValueError
        Shapes/labels are incompatible or the full interval is too short to
        supply final bootstrap draws.

    Notes
    -----
    Use fresh reproducible draws at each completed prefix, shared by all
    methods. Keep early bounds NaN before MIN_CI_N. Hold forecasts fixed and
    refit the unregularized comparator inside every draw; no retraining or
    hyperparameter reselection. Save interval arrays before any plotting.
    Use one fitted vector over the entire resampled prefix for each oracle.
    """


    return


def scorecard_intervals(point_table, bootstrap, phase):
    """
    Combine final point estimates and their paired empirical intervals.

    Parameters
    ----------
    point_table : pandas.DataFrame
        One named method per index row, with columns specified by METRIC_KEYS.
    bootstrap : dict
        Output of bootstrap_prefix_metrics for those methods and scored dates.
    phase : str
        Prefix for scorecard interval registry keys.

    Returns
    -------
    table : pandas.DataFrame
        MultiIndex (Method, Metric), with Estimate, Lower 95%, Upper 95% columns.
        Register each interval and retain the point_table values unchanged.

    Notes
    -----
    P95 is an empirical percentile; maximum-error bounds are resampling
    ranges, not valid extrapolations to unseen tail events.
    """


    return


def check_bootstrap_oracles(X_score, y_score, seed=BOOT_SEED):
    """
    Check weighted and batched comparator values against the scalar solver.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features for a small scored interval.
    y_score : ndarray, shape (n,)
        Matching targets.
    seed : int, default BOOT_SEED
        Reproducible bootstrap-check seed.

    Returns
    -------
    None
        Raise AssertionError if weighted/bootstrap losses differ from scalar
        whole-draw constrained optima. Include singular and active cases.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## II.6 — Choose settings by minimum validation regret / time

## Exercise 9 — Tune without consulting the future

Compare the specified $\lambda$ grid for FTRL and $\eta_0$ grid for OGD. Select the setting
with minimum **final validation $R_{180}/180$**, not the smallest value reached at an arbitrary
intermediate prefix. Why is this ranking equivalent to ranking validation mean squared loss?

Define the candidate label $h$ and explain the superscript $(h)$ in $R_{180}^{(h)}$.
Use only the declared grids and deterministic tie-breaking. Interpret step-size/penalty sensitivity,
the symmetric-log scale and the scope of the pointwise bootstrap bands. Record the validation-selected
method separately from any later test winner; do not change the grids after viewing test results.


### Your answer — Exercise 9

In [ ]:
def select_validation_settings(X_val, y_val, validation_oracle, axis_labels=None):
    """
    Search the declared grids and freeze settings using final validation average regret.

    Parameters
    ----------
    X_val : ndarray, shape (N_VAL, d)
        Validation-only chronological features.
    y_val : ndarray, shape (N_VAL,)
        Validation-only targets.
    validation_oracle : ndarray, shape (N_VAL,)
        Matching unregularized fixed-comparator prefix losses.
    axis_labels : iterable or None, default None
        Optional validation date labels for saved intervals.

    Returns
    -------
    selection : dict
        Keys: validation_runs ((method, value) -> prequential result),
        validation_table (DataFrame), selected (method -> float),
        validation_champion (Series), validation_bootstrap (dict),
        validation_curve_ci (method -> metric -> (N_VAL, 2)),
        validation_grid_ci (method -> (n_candidates, 2)),
        validation_metrics_ci (DataFrame), and figure (Matplotlib figure).
        The table contains Method, Value, Mean loss, Validation R/n,
        Validation RMSE (rentals), and Selected within method.

    Notes
    -----
    Use fresh calibration-initialized learners for FTL's single value 0.0,
    FTRL_GRID and OGD_GRID. Rank final signed R_180/180, breaking within-method
    ties by smaller Value. Identify the validation champion with deterministic
    Method/Value tie-breaking. Compute paired bootstrap arrays before plotting.
    Show searched grids, selected settings and pointwise intervals; use a
    symmetric-log regret axis. Do not access test targets or expand the grids.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

In [ ]:
def plot_validation_curves(validation_runs, selected, validation_oracle, validation_bootstrap):
    """
    Plot validation mean loss and signed average regret at the selected settings.

    Parameters
    ----------
    validation_runs : dict
        Prequential results keyed by (method, value).
    selected : dict
        Frozen within-method settings selected on final validation regret.
    validation_oracle : ndarray, shape (N_VAL,)
        Best fixed cumulative loss for every validation prefix.
    validation_bootstrap : dict
        Saved paired bootstrap output for all validation candidates.

    Returns
    -------
    figure : matplotlib.figure.Figure
        Two panels: average loss with the fixed comparator, and signed R_n/n.
        Plot saved pointwise bands with opacity CI_ALPHA and missing early bounds.

    Notes
    -----
    Include FTL, FTRL and OGD with distinct line styles. Use candidate labels
    consistent with select_validation_settings; do not retune at each prefix.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

# Part III — Final metrics and the limits of regret / time

## III.1 — Freeze choices; evaluate on the chronological test stream

## Exercise 10 — Continue learning, but stop tuning

Start fresh model states, replay the validation period at the frozen settings, and then score
test days. Check that this produces the same state as uninterrupted execution. Add three
simple reference predictors: the frozen calibration fit, yesterday's count and last week's count.

Show equivalence with uninterrupted validation-plus-test execution. Establish whether the three fixed baselines
are feasible in the chosen representation and distinguish observed lags from recursive forecasts.


### Your answer — Exercise 10

In [ ]:
def evaluate_frozen_models(selected, validation_runs, X_val, y_val, X_test, y_test, feature_names):
    """
    Replay validation at frozen settings, then score the chronological test stream.

    Parameters
    ----------
    selected : dict
        Frozen primary-method hyperparameters.
    validation_runs : dict
        Recorded candidate runs used to verify deterministic replay.
    X_val : ndarray, shape (N_VAL, d)
        Validation features, replayed as observed history.
    y_val : ndarray, shape (N_VAL,)
        Validation outcomes.
    X_test : ndarray, shape (n_test, d)
        Later test features.
    y_test : ndarray, shape (n_test,)
        Test outcomes revealed only after their predictions.
    feature_names : list[str]
        Names aligned with feature columns, including intercept and demand lags.

    Returns
    -------
    evaluation : dict
        Keys: test_runs (six named prequential results), baseline_vectors
        (three feasible fixed coefficient arrays), and replay_seconds (method -> float).
        Baseline names: "Frozen calibration", "Previous day", "Previous week".

    Notes
    -----
    Build fresh primary states, replay validation and continue without a
    clock reset. Verify equivalence with uninterrupted validation+test runs.
    Use workspace theta_ref and FEATURE_SCALE to define fixed baselines in
    the same feature coordinates. Lag baselines use observed past counts,
    not recursively generated multi-step forecasts. Verify their feasibility.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Exercise 11 — Compute a final scorecard and prefix curves

Fit the hindsight vector on **test days only** and compute total regret, regret per test day,
RMSE, MAE, signed bias, and a tail-error metric. Why must the oracle be unregularized even
when the learner is FTRL?

State all units and the sign convention for bias. Check the conversion between normalized loss and RMSE.
Attach paired intervals with a refitted comparator to the scorecard and plot the test-prefix curves.
Compare OGD with the daily and weekly references using paired RMSE-reduction draws. Write your own
final interpretation from the actual completed results; do not assume any method must win.


### Your answer — Exercise 11

In [ ]:
def evaluate_test_metrics(X_test, y_test, test_runs, baseline_vectors, axis_labels=None):
    """
    Compute the final independent test scorecard and matching empirical intervals.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test-only features.
    y_test : ndarray, shape (n_test,)
        Test-only normalized outcomes.
    test_runs : dict
        Six causal prequential results; add regret and average_regret arrays
        to each run only after predictions have been completed.
    baseline_vectors : dict[str, ndarray]
        Three fixed feasible baseline coefficient vectors.
    axis_labels : iterable or None, default None
        Optional test date labels for saved interval bounds.

    Returns
    -------
    scores : dict
        Keys: metrics (DataFrame indexed by Method), metrics_ci (DataFrame),
        test_oracle_losses (n_test,), theta_test_oracle (d,), oracle_direct_loss
        (float), oracle_seconds (float), test_bootstrap (dict), test_curve_ci (dict).
        metrics contains Days and every display column specified by METRIC_KEYS.

    Notes
    -----
    Fit the comparator only on test dates with no penalty. Verify the full
    fit agrees with the final prefix optimum and does not lose to any feasible
    fixed baseline. Preserve signed adaptive regret. RMSE, MAE, bias, P95 and
    maximum errors are in rental units; bias is prediction minus observation.
    Refit comparators in paired bootstrap draws and save all bounds before plots.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

In [ ]:
def plot_test_curves(test_runs, test_oracle_losses, test_bootstrap):
    """
    Plot test cumulative loss, static regret and regret per scored day.

    Parameters
    ----------
    test_runs : dict
        Results including regret and average_regret for FTL, FTRL and OGD.
    test_oracle_losses : ndarray, shape (n_test,)
        Cumulative fixed-comparator optima for test prefixes.
    test_bootstrap : dict
        Previously saved learner and oracle prefix interval arrays.

    Returns
    -------
    figure : matplotlib.figure.Figure
        Three panels with aligned test-prefix axes and pointwise interval bands.
        Include the prefix oracle in the cumulative-loss panel and preserve
        negative regrets in the other panels.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

In [ ]:
def compare_test_models(metrics, metrics_ci, test_bootstrap, selected, validation_champion):
    """
    Summarize the observed test comparison without changing the selection rule.

    Parameters
    ----------
    metrics : pandas.DataFrame
        Final six-model test scorecard.
    metrics_ci : pandas.DataFrame
        Saved final metric intervals indexed by (Method, Metric).
    test_bootstrap : dict
        Paired final bootstrap draws aligned across all models.
    selected : dict
        Frozen validation-selected settings.
    validation_champion : pandas.Series
        Primary model chosen on validation, not chosen retrospectively on test.

    Returns
    -------
    comparison : dict
        Keys: summary (DataFrame of observed comparisons) and rmse_gain_ci
        (daily/weekly baseline name -> two-element interval of percentage RMSE
        reduction for OGD). Register the paired comparison bounds.

    Notes
    -----
    Compute conclusions from completed results, not from a preferred winner.
    Distinguish the validation-selected method from the best realized test
    method. Use paired draws for percentage RMSE comparisons and explain
    selection/conditional-interval limits in your Markdown answer.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## III.2 — Errors over time, tail behaviour and computational cost

## Exercise 12 — Look beyond the global average

Plot the full trajectory with period boundaries, the last 120 test forecasts, rolling RMSE,
and OGD's largest errors. Inspect coefficient norms and update magnitudes. Does decreasing
average regret imply every recent prediction is improving?

Add 28-day mean coefficient norms and update magnitudes with saved bootstrap bands and faint raw paths.
Repeat fresh runs to estimate timing uncertainty separately from calendar resampling. Explain online state
memory, logging overhead, per-update work and the separate cost of hindsight evaluation.


### Your answer — Exercise 12

In [ ]:
def plot_forecast_diagnostics(daily, dates, cal_slice, val_slice, test_slice, y_test, test_runs):
    """
    Inspect realized forecasts, recent errors and large test mistakes after selection.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw observations; all-period inspection is allowed only now.
    dates : pandas.DatetimeIndex
        Usable feature dates.
    cal_slice : slice
        Initial calibration interval.
    val_slice : slice
        Validation interval.
    test_slice : slice
        Final test interval.
    y_test : ndarray, shape (n_test,)
        Normalized test outcomes.
    test_runs : dict
        Recorded causal forecasts for primary methods and fixed baselines.

    Returns
    -------
    diagnostics : dict
        Keys: figure (three-panel Figure), worst_errors (top-eight OGD error
        DataFrame), rolling_rmse (n_test, 4) and rolling_rmse_ci (n_test, 4, 2).
        Rolling columns: FTL, FTRL, OGD, Previous day. Register interval arrays.

    Notes
    -----
    Show the whole trajectory with split shading, the final 120 forecasts,
    and 30-day trailing RMSE with paired bootstrap bands. Add no confidence
    band to a single recorded forecast/outcome. Do not feed diagnostics back
    into hyperparameter selection or infer disruption causes from counts alone.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

In [ ]:
def diagnose_stability_and_cost(test_runs, selected, X_val, y_val, X_test, y_test, test_dates):
    """
    Measure coefficient stability and repeated fresh-run computational cost.

    Parameters
    ----------
    test_runs : dict
        Recorded primary-model coefficient and movement paths.
    selected : dict
        Frozen settings for FTL, FTRL and OGD.
    X_val : ndarray, shape (N_VAL, d)
        Validation features for fresh timing replays.
    y_val : ndarray, shape (N_VAL,)
        Validation outcomes.
    X_test : ndarray, shape (n_test, d)
        Test features.
    y_test : ndarray, shape (n_test,)
        Test outcomes.
    test_dates : pandas.DatetimeIndex
        Dates aligned with recorded state paths.

    Returns
    -------
    diagnostics : dict
        Keys: figure (norm/movement Figure), cost_table (DataFrame),
        timing_samples (N_TIMING_REPEATS, 3, 2), oracle_seconds (float),
        oracle_time_ci (2,), norm_mean_ci (n_test, 3, 2), and step_mean_ci
        (n_test, 3, 2). Save the empirical interval arrays.

    Notes
    -----
    Show 28-day trailing means with paired block bands and faint raw state
    traces. Use repeated fresh learners for timing, with separate replay/test
    means and bootstrap intervals over timing repetitions, not calendar days.
    Measure common hindsight evaluation separately from online update costs.
    Explain state memory, logging overhead and dimension-dependent work.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

### Pooled RMSE across the whole forecast period, with 95% empirical intervals

This requested bar chart includes **all 634 forecast days**: validation and test,
excluding calibration. Compute `sqrt(mean(squared error))` over that complete period;
do **not** average rolling RMSEs or one-day absolute errors. All six models use the
same dates and paired block draws. Because validation helped select settings, this
is descriptive; the independent chronological test scorecard remains Exercise 11.


### Your interpretation

In [ ]:
def pooled_forecast_rmse(validation_runs, selected, test_runs, baseline_vectors, X_val, y_val, y_test, axis_labels=None):
    """
    Compare pooled RMSE over validation and test, excluding calibration.

    Parameters
    ----------
    validation_runs : dict
        Recorded validation candidate runs.
    selected : dict
        Frozen settings identifying each primary validation path.
    test_runs : dict
        Six recorded test forecast paths.
    baseline_vectors : dict[str, ndarray]
        Fixed feasible reference vectors used on both intervals.
    X_val : ndarray, shape (N_VAL, d)
        Validation features for fixed-baseline predictions.
    y_val : ndarray, shape (N_VAL,)
        Validation targets.
    y_test : ndarray, shape (n_test,)
        Test targets.
    axis_labels : iterable or None, default None
        Optional six labels for registered model-level RMSE bounds.

    Returns
    -------
    pooled : dict
        Keys: forecast_rmse_summary (six-row DataFrame), forecast_rmse_ci
        (6, 2), forecast_predictions (method -> 634 normalized forecasts),
        and figure (bar chart with empirical interval endpoints).

    Notes
    -----
    Pool all 634 forecast errors before taking RMSE; do not average rolling
    RMSEs or one-day absolute errors. Pair block draws across models and plot
    all interval endpoints, which need not be symmetric around point estimates.
    This is descriptive because validation participated in selection; it does
    not replace the independent final-test scorecard.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## III.3 — Check a finite-time OGD bound without overclaiming

## Exercise 13 — Connect the implementation to the course guarantee

Use $\|\theta\|\le B=4$, $\|x_t\|\le1$, $|y_t|\le1$.
Find $D$ and $G$. The course derives a constant-step bound. What changes for our decreasing
schedule, and may we infer a logarithmic rate because the loss is quadratic?

Derive a bound valid with the validation offset in the test clock, then compare its uniform-gradient and
realized-gradient forms with observed test regret. State the exact one-sided asymptotic no-regret implication,
and discuss negative regret, monotonicity, and what a finite curve can establish. Do not assign statistical
confidence intervals to exact analytical inequalities.


### Your answer — Exercise 13

In [ ]:
def check_ogd_bound(X_test, y_test, ogd_run, eta0, offset=N_VAL):
    """
    Evaluate the finite-horizon OGD bounds derived in your written answer.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test features satisfying the stated norm assumption.
    y_test : ndarray, shape (n_test,)
        Test outcomes satisfying the stated outcome assumption.
    ogd_run : dict
        Recorded OGD result including pre-update theta, predictions and loss.
    eta0 : float
        Positive validation-selected step scale.
    offset : int, default N_VAL
        Number of scored updates already completed before test; do not reset it.

    Returns
    -------
    bound_table : pandas.DataFrame
        Rows for observed static regret, the bound using observed gradients and
        the bound using a uniform gradient bound. Columns: Cumulative and
        Per test day. Exact theoretical bounds have no sampling intervals.

    Notes
    -----
    Derive D, G and the nonconstant-step inequality yourself before coding.
    Verify the step schedule is positive/nonincreasing, gradients respect the
    uniform bound, and empirical regret is no larger than the derived bounds.
    Use the post-validation feasible initial state and a test-only comparator.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## III.4 — Why static average regret is not enough

## Exercise 14 — Build a zero-regret but inaccurate predictor

Let $K=[0,1]$, $\widehat y_t=1/2$ fixed before observing anything, and let the stream be
200 zeros followed by 200 ones. Compute its static average regret and mean squared loss.
Compare with a hindsight benchmark allowed one coefficient on each 200-day block.

Also evaluate the same predeclared predictor on an all-zero stream of the same length. Explain what this
constructed example does and does not establish about universal guarantees or the real 13-feature model.


### Your answer — Exercise 14

In [ ]:
def analyze_toy_stream(block_length=200, prediction=0.5):
    """
    Evaluate the fixed scalar predictor on a two-regime stream and a second stream.

    Parameters
    ----------
    block_length : int, default 200
        Positive number of zeros followed by the same number of ones.
    prediction : float, default 0.5
        Fixed prediction declared before any outcomes; feasible set is [0, 1].

    Returns
    -------
    results : pandas.Series
        Static regret per day, mean half squared loss, MAE, two-block regret
        per day, and the same predictor's regret per day on an all-zero stream.
    figure : matplotlib.figure.Figure
        Deterministic outcomes and the fixed prediction, with no sampling bands.

    Notes
    -----
    Fit one fixed scalar hindsight comparator for the full balanced stream
    and one per known block for the stronger reference. Evaluate the same
    predeclared predictor on the all-zero stream as a separate check. Compute
    all values rather than hard-coding them; interpret them in Markdown.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Exercise 15 — Strengthen the real-data benchmark

On the actual test interval, partition the days into fixed consecutive 13-week (91-day) blocks (last
block possibly shorter). Fit one coefficient vector in $K$ per block. Compare average excess
loss against this richer oracle with static average regret. Does the original theorem cover it?

Keep the boundaries fixed. For empirical intervals, resample within the predetermined blocks and refit both
the static and blockwise comparators on the same draw. Explain what extra flexibility can and cannot show
about coefficient drift, and state a possible comparator-variation constraint for a dynamic guarantee.


### Your answer — Exercise 15

In [ ]:
def blockwise_tracking(X_test, y_test, test_runs, test_dates, block_size=91):
    """
    Compare static regret with excess loss against a predetermined blockwise oracle.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test-only observed feature rows.
    y_test : ndarray, shape (n_test,)
        Test-only normalized targets.
    test_runs : dict
        Completed primary-model forecast/loss paths.
    test_dates : pandas.DatetimeIndex
        Aligned chronological test dates.
    block_size : int, default 91
        Predetermined consecutive block length, with the last block shorter if needed.

    Returns
    -------
    tracking : dict
        Keys: block_table (dates, sizes, oracle mean losses), tracking
        (method-indexed static and blockwise regret/day DataFrame),
        block_ci (DataFrame), and tracking_metrics_ci (DataFrame).
        Register block and model comparison intervals.

    Notes
    -----
    Fit one feasible unregularized fixed vector per block. Verify blockwise
    oracle loss never exceeds global static oracle loss. For intervals, keep
    block boundaries fixed and resample within each block, pairing all methods
    and refitting both comparators on the same combined draw. Do not optimize
    boundaries using targets or treat this hindsight oracle as deployable.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## III.5 — Other limits: the loss, information and deployment context

## Exercise 16 — What if underprediction is three times more costly?

Suppose a hypothetical one-period decision $q\in[0,Q_{\max}]$ has cost
$$
C(q,y)=c_h(q-y)_++c_b(y-q)_+,\qquad c_h=1,\ c_b=3.
$$
Which conditional functional should be predicted? Derive a valid subgradient in $q$.
Would our squared-loss regret certify good performance under this cost?

Include the equality case of the subgradient and the feasibility constraints for a linear decision.
Explain the appropriate comparator and whether the observed rental counts support a literal inventory interpretation.
The Python cell below is a small implementation companion to your derivation, not a new bike forecasting model.


### Your answer — Exercise 16

In [ ]:
def inventory_cost(q, target, holding_cost=1.0, shortage_cost=3.0):
    """
    Evaluate the asymmetric one-period surplus/shortage loss from Exercise 16.

    Parameters
    ----------
    q : float
        Feasible nonnegative decision quantity.
    target : float
        Observed demand in the same units.
    holding_cost : float, default 1.0
        Positive cost per unit of surplus.
    shortage_cost : float, default 3.0
        Positive cost per unit of unmet demand.

    Returns
    -------
    cost : float
        The stated piecewise-linear inventory loss, not a squared error.
    """


    return


def inventory_subgradient(q, target, holding_cost=1.0, shortage_cost=3.0):
    """
    Choose a valid scalar subgradient of inventory_cost with respect to q.

    Parameters
    ----------
    q : float
        Current decision quantity.
    target : float
        Revealed demand.
    holding_cost : float, default 1.0
        Positive surplus unit cost.
    shortage_cost : float, default 3.0
        Positive shortage unit cost.

    Returns
    -------
    subgradient : float
        A valid derivative away from equality and a valid selected subgradient
        at equality; state your equality convention in the written answer.
    """


    return


def projected_inventory_update(q, target, eta, capacity, holding_cost=1.0, shortage_cost=3.0):
    """
    Perform one feedback-based inventory update within the feasible quantity interval.

    Parameters
    ----------
    q : float
        Current quantity in [0, capacity].
    target : float
        Newly observed demand.
    eta : float
        Positive update step.
    capacity : float
        Positive upper bound on feasible quantity.
    holding_cost : float, default 1.0
        Surplus unit cost.
    shortage_cost : float, default 3.0
        Shortage unit cost.

    Returns
    -------
    q_next : float
        Updated quantity in [0, capacity], using your selected subgradient.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Exercise 17 — Explain the ridge distinction

If we define $\widetilde\ell_t(\theta)=\ell_t(\theta)+\gamma\|\theta\|^2/2$, $\gamma>0$,
what changes in the gradient, comparator and regret theorem? Is that the same as our FTRL?

Account for the intercept, the length of the scored interval and the learning-rate assumptions.
Explain the difference between a penalty included in every evaluated loss and one fixed algorithmic penalty
on cumulative unregularized losses. Implement the documented ridge-loss helpers separately from the main experiment.


### Your answer — Exercise 17

In [ ]:
def ridge_loss(theta, x, target, gamma):
    """
    Evaluate the per-observation ridge-regularized loss defined in Exercise 17.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficients, including the intercept coordinate.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.
    gamma : float
        Positive penalty weight applied to every observation and coordinate.

    Returns
    -------
    loss : float
        Regularized per-round loss; unlike the main experiment, evaluation
        includes the per-round penalty.
    """


    return


def ridge_gradient(theta, x, target, gamma):
    """
    Evaluate the gradient of the per-round ridge loss.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficients, including the intercept.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized target.
    gamma : float
        Positive per-round penalty weight.

    Returns
    -------
    gradient : ndarray, shape (d,)
        Gradient in the same scaled coordinate system as theta.
    """


    return


def fit_ridge_comparator(X_score, y_score, gamma):
    """
    Fit the fixed feasible comparator for a sum of per-round ridge losses.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features from exactly the scored interval.
    y_score : ndarray, shape (n,)
        Aligned normalized outcomes.
    gamma : float
        Positive penalty weight appearing in every scored loss.

    Returns
    -------
    theta : ndarray, shape (d,)
        Feasible minimizer of the summed ridge_loss objectives. Account for
        interval length when assembling the cumulative penalty.

    Notes
    -----
    This comparator belongs to a different evaluated loss from the main
    FTL/FTRL/OGD experiment; explain the distinction before implementing it.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Saved empirical intervals — inspect or reuse the exact plotted bounds

Build the arrays **before** their plots. Complete the export function below for every
interval and its matching index/axis label, plus the tabular scorecards:

- `validation_grid_ci[method]`: one `[lower_95, upper_95]` per tried hyperparameter;
- `validation_curve_ci[method]["mean_loss" or "average_regret"]`: one row per validation prefix;
- `test_curve_ci[method]["total_loss", "regret", "average_regret", ...]`: one row per test prefix;
- `metrics_ci`: every final-test performance statistic, including all six models;
- `forecast_rmse_ci`: one row per model for the all-forecast-period bar chart;
- `CI_BOUNDS`: a registry also containing EDA, local-window, timing and tracking intervals.

For example, `test_curve_ci["OGD"]["average_regret"][n - 1]` is the saved interval
after exactly `n` test forecasts. Bounds are stored as `NaN` before four weekly blocks
are available; no future data are borrowed to create an early narrow band.
The NPZ files load with `allow_pickle=False`; the CSV gives a readable indexed view.


### Your interpretation

In [ ]:
def export_empirical_intervals(results_dir, tables, manifest):
    """
    Validate and export student-computed interval arrays without overwriting instructor results.

    Parameters
    ----------
    results_dir : pathlib.Path
        Use a dedicated student directory, for example
        online_learning_bike_sharing_student_results/en or its fr counterpart.
    tables : dict[str, pandas.DataFrame]
        Named final/validation/pooled/tracking/timing scorecards to export as CSV.
    manifest : dict
        Record CI_LEVEL, N_BOOT, BLOCK_LENGTH, BOOT_SEED, MIN_CI_N, CI_ALPHA,
        fixed-forecast/refitted-oracle semantics, pointwise limitations and timing
        resampling details. State the descriptive status of pooled forecast RMSE.

    Returns
    -------
    files : dict[str, pathlib.Path]
        Include empirical_95_ci_bounds.npz, empirical_95_ci_axes.npz,
        empirical_95_ci_bounds.csv, empirical_95_ci_manifest.json and one CSV
        per supplied scorecard. No object arrays or pickle.

    Notes
    -----
    Read CI_BOUNDS and CI_AXES. Validate shape, endpoint order, early NaNs,
    matching labels and prefix loss/regret scaling. The registries contain
    endpoints only: perform draw-level comparator checks upstream in the
    oracle and prequential verification functions, not in this exporter.
    Build and check all arrays before export. Reload NPZ with allow_pickle=False
    and verify equality. Do not load instructor result files as exercise answers.
    Keep English and French student exports in separate directories.
    """


    return


**Execution workspace:** call your completed functions, unpack their outputs, and run your checks here.

## Final synthesis — Write your own conclusions

Complete a table with three columns: **observed result**, **what it supports**, and **what it does not establish**.
Address low test average regret, a no-regret upper bound, a fixed comparator, grid-based selection,
bounded convex losses, squared-loss performance, and a favorable finite curve.
Discuss the limits of this two-year, one-city stream, daily feedback availability, demand censoring,
changing conditions, alternative tracking schemes and repeated test-set tuning.
Do not copy a completed conclusion: use your own executed results and two distinct deployment limitations.

### Completion checklist

- [ ] Open dataset with attribution, original README and verified local cache.
- [ ] Calibration-only descriptive analysis and strictly causal features.
- [ ] Common convex loss, bounded feasible set and explicit feedback protocol.
- [ ] FTL, full-loss FTRL and OGD to implement and numerically check.
- [ ] Exact constrained hindsight prefix benchmark, not a changing-oracle shortcut.
- [ ] Minimum validation regret/day selects frozen settings; test remains sequential.
- [ ] Final accuracy, regret, tail-error, stability and cost diagnostics.
- [ ] Paired empirical 95% pointwise intervals, saved bounds and an all-forecast RMSE chart.
- [ ] Finite-time bound, weak-benchmark counterexample and stronger tracking diagnostic.
- [ ] Ridge and asymmetric-loss distinctions answered.

### Final student write-up (one page)

State the loss and information set; describe the split and scaling; report the validation-selected
settings and final scorecard; interpret one error plot; and explain **two distinct reasons** why
low static average regret is not enough for deployment. Use the executed tables rather than
claiming that a preferred algorithm must win.

### References and reproducibility

1. Paul Minchella, *Online Learning*, [course slides](../Cours_Online_Learning.pdf), especially printed
   slides 26–29, 32–49 and 53–65.
2. Hadi Fanaee-T, *Bike Sharing*, UCI Machine Learning Repository,
   [doi:10.24432/C5W894](https://doi.org/10.24432/C5W894),
   [dataset page](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), CC BY 4.0.
3. Fanaee-T and Gama (2013),
   [doi:10.1007/s13748-013-0040-3](https://doi.org/10.1007/s13748-013-0040-3).
4. Elad Hazan, *Introduction to Online Convex Optimization*, for regret proofs and extensions.

The seed controls only numerical checks, not the observations. There is no shuffled split.
With fixed package versions the data, settings and predictions are deterministic up to numerical
precision; runtime measurements are not. Record your package versions in Part 0.
**Restart the kernel and Run All** after completing your implementation. Keep the notebook and
`online_learning_bike_sharing_data` directory together for an offline rerun.

**Bootstrap reproducibility.** The resampling seed, block length, repetitions, minimum
history and pointwise-coverage caveats are recorded in
`online_learning_bike_sharing_student_results/en/empirical_95_ci_manifest.json`. Point estimates
are unchanged by adding intervals. Empirical bands are not confidence sequences and do
not certify 95% simultaneous or distribution-free coverage under drift.


### Your interpretation